In [ ]:
# Dont delete this - this is necessary to run it on colab
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  %cd /content/drive/MyDrive/Python Code/Paper/scripts/fluxes/wrtds

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1jKcNrw1UEQ_p8vbILfPLdYnIma0uaqCy/Python_Code/Paper/scripts/fluxes/wrtds


In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib as mpl
import seaborn as sns
import warnings
from matplotlib import pyplot as plt
from matplotlib import dates as mdates
from matplotlib import colors as mcolors
import os

In [ ]:
chemicals = ["TN", "TP", "Fe", "dSi", "TSS"]
rivers = ["Puelo", "Yelcho", "Palena", "Cisnes", "Aysen"]
#rivercolors = ["blue", "orange", "yellow", "purple", "red"]
#rivercolors = ["#0072B2", "#E69F00", "#009E73", "#F0E442", "#D55E00"]
#rivercolors = sns.color_palette("muted", n_colors=len(rivers)) #most recent as prior to Apr 25 2025
rivercolors = ["#f3d658", "#e1c9a8", "#e8a3cb", "#a6b6d2", "#f9a68a"]
vivid_colors = ['#ffde4c', '#eccb9d', '#f695cd', '#9db4db', '#ffa384'] #adjust_saturation(c, sat_factor=1.4, light_factor=1.0)
point_colors = ['#e8bd00', '#ff9d14', '#ff169c', '#286cdf', '#ff4c10'] #adjust_saturation(c, sat_factor=1.6, light_factor=0.7)
streamflow_colors = ["gray"] * 5
#streamflow_colors = rivercolors

In [ ]:
wrtds_data_root = os.getenv("WRTDS_DATA_ROOT", "../../../data/fluxes/WRTDS/workspace_scaflow/")
flow_data_path = os.getenv("FLOW_DATA_PATH", "../../../data/streamflow/est_outlet_flow/Downstream_Estimate_All_Rivers_rolling60.csv")
areas_path = "../../../data/geospatial/stats/watershed_area_stats.csv"
csv_out_root = os.getenv("CSV_OUT_ROOT", "../../../data/fluxes/WRTDS/totals_scaflow/")
fig_out_root = os.getenv("FIG_OUT_ROOT", "../../../figures/chemistry/flux/WRTDS_scaflow/")

In [ ]:
areas = pd.read_csv(areas_path).set_index("river_name").loc[rivers]["total_area"]
areas

,total_area
river_name,
Puelo,9121.633249
Yelcho,11369.277107
Palena,13147.563085
Cisnes,5116.698637
Aysen,12181.393611


In [ ]:
trim_to_water_year=True
alternate_percentile=False
add_fluxbiases=False
water_year=(pd.to_datetime("1-4-2022", dayfirst=True), pd.to_datetime("31-3-2023", dayfirst=True))

In [ ]:
def read_csv(river, chem, pathf):
  df = pd.read_csv(pathf.format(r=river,c=chem))
  if "Date" in df.columns:
    df.set_index(pd.to_datetime(df["Date"]), inplace=True)
  df.drop(columns=["Date", "Unnamed: 0"], errors="ignore", inplace=True)
  da = xr.DataArray(df)
  return da

def combine_results(rivers, chemicals, pathf, apply=lambda df: df):
  return xr.combine_nested(
      [
          [
              apply(read_csv(r, c, pathf))
              for c in chemicals
          ]
          for r in rivers
      ],
      concat_dim=["river", "chem"]
      ).assign_coords({"river": rivers, "chem": chemicals})

In [ ]:
all_daily_results = combine_results(rivers, chemicals, wrtds_data_root+"{r}/{c}/Results/{r}_daily{c}_WRTDS.csv")
date_idx = all_daily_results["Date"]

all_samples = combine_results(rivers, chemicals, wrtds_data_root+"/{r}/{c}/{r}_{c}_WRTDS.csv", apply=(lambda da: da.isel(dim_1=1)))

all_dailyboot = combine_results(rivers, chemicals, wrtds_data_root+"/{r}/{c}/Results/dailyBootOut.csv").rename({"dim_0": "Date"}).assign_coords(Date=date_idx)

all_dayPctConc = combine_results(rivers, chemicals, wrtds_data_root+"/{r}/{c}/Results/dayPct_conc.csv") # not used yet
#all_dayPctFlux = combine_results(rivers, chemicals, wrtds_data_root+"/{r}/{c}/Results/dayPct_flux.csv")


# Reading flow data directly from the source rather than from individual WRTDS folders
all_flows = xr.DataArray(pd.read_csv(flow_data_path).set_index("Date", drop=True)).rename({"dim_1": "river"}) # TODO: Draw flow data from the WRTDS folders themselves?
all_flows["Date"] = pd.to_datetime(all_flows["Date"])

all_errorstats = combine_results(rivers, chemicals, wrtds_data_root+"/{r}/{c}/Results/{r}_ErrorStats_WRTDS.csv").sum(dim="dim_0")

all_flows_smoothed=all_flows.rolling(Date=14).mean()

if trim_to_water_year: #TODO: combine into dataset
  all_daily_results = all_daily_results.sel(Date=slice(*water_year))
  all_samples = all_samples.sel(Date=slice(*water_year))
  all_dailyboot = all_dailyboot.sel(Date=slice(*water_year))
  all_dayPctConc = all_dayPctConc.sel(Date=slice(*water_year))
  #all_dayPctFlux = all_dayPctFlux.sel(Date=slice(*water_year))
  all_flows = all_flows.sel(Date=slice(*water_year))
  all_flows_smoothed = all_flows_smoothed.sel(Date=slice(*water_year))
  date_idx = all_daily_results["Date"]

/tmp/ipython-input-3696465641.py:10: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'Date' ('Date',) The recommendation is to set join explicitly for this case.
  return xr.combine_nested(
/tmp/ipython-input-3696465641.py:10: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'Date' ('Date',) The recommendation is to set join explicitly for this case.
  return xr.combine_nested(
/tmp/ipython-input-3696465641.py:10: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to 

# GenFlux is the WRTDS-Kalman estimate of the flux for each day. FluxDay is the WRTDS etimate
# ConcDay is the WRTDS estimate for the concentration for each day. GenConc is the WRTDS-Kalman estimate. GenConc almost exactly matches the sampled values on the sampled dates

In [ ]:
boot_qs = all_dailyboot.quantile([0.05, 0.95], dim="dim_1")
boot_qs = xr.concat(
    [boot_qs.sel(quantile=0.05, drop=True),
     all_daily_results.sel(dim_1="GenFlux", drop=True),
     boot_qs.sel(quantile=0.95, drop=True)],
          dim="quantile").assign_coords(quantile=["p5", "actual", "p95"]) # Dis some bullshit
#boot_qs = all_dayPctFlux.sel(dim_1=["p5", "p50", "p95"]).rename({"dim_1": "quantile"}).assign_coords(quantile=["0.05", "actual", "0.95"]) # TODO: turn it back to dim_1


cumboot_qs = all_dailyboot.cumsum(dim="Date").quantile([0.05, 0.95], dim="dim_1")
cumboot_qs = xr.concat(
    [cumboot_qs.sel(quantile=0.05, drop=True),
     all_daily_results.sel(dim_1="GenFlux", drop=True).cumsum(dim="Date"),
     cumboot_qs.sel(quantile=0.95, drop=True)],
          dim="quantile").assign_coords(quantile=["p5", "actual", "p95"]) # Dis some bullshit
#cumboot_qs = all_dailyboot.cumsum(dim="Date").quantile([0.05, 0.5, 0.95], dim="dim_1").assign_coords(quantile=["0.05", "actual", "0.95"])

#TODO: Alternate percentiles?

# Convert kg to tonnes
boot_qs /= 1000
cumboot_qs /= 1000
all_dailyboot /= 1000

In [ ]:
# # # Sanity check: how different is dayPct from the directly calculated quantiles of dailyboot?
# _ = all_dailyboot.quantile([0.05, 0.95], dim="dim_1")
# (((_ - all_dayPctFlux.sel(dim_1=["p5", "p95"]).assign_coords(dim_1=[0.05, 0.95]).rename({"dim_1": "quantile"})) / _) > 0.01).sum()

In [ ]:
# # Sanity check: is dayPct_flux the same as the bootstrap stuff?
# assert (all_dailyboot.quantile(0.5) == boot_qs.sel(quantile="actual")).any()
# # PASSED

In [ ]:
# Sanity check: does CumulativeGenFux start at 0? NOTE: No it does not, however cumsum(GenFlux) is the same as CumulativeGenFlux minus the stuff before the water year
#all_daily_results.sel(dim_1="CumulativeGenFlux").isel(Date=0).to_pandas()

In [ ]:
#all_flows_smoothed=all_flows.rolling(Date=14).mean()

total_fluxes = {
    c: all_daily_results.sel(dim_1="GenFlux", chem=c).values.sum()
    for c in chemicals
    }

In [ ]:
season_codes = ((np.ceil(date_idx.dt.month / 3) + 2) % 4).astype("int").rename("season")
season={0: "Fall", 1: "Winter", 2: "Spring", 3: "Summer"}
cumboot_qs.groupby(season_codes).last().assign_coords(season=["Fall", "Winter", "Spring", "Summer"]) # For my anxiety

<xarray.DataArray (quantile: 3, chem: 5, river: 5, season: 4)> Size: 2kB
array([[[[5.33662207e+02, 1.01066982e+03, 1.33863200e+03,
          1.62132323e+03],
         [4.09659036e+02, 7.70144434e+02, 1.43381147e+03,
          2.67058330e+03],
         [8.92613427e+02, 1.60181016e+03, 2.67133227e+03,
          3.19207554e+03],
         [3.59938865e+02, 7.14939806e+02, 1.89109731e+03,
          2.20800385e+03],
         [6.33356441e+02, 1.25353918e+03, 1.90756382e+03,
          2.28955363e+03]],

        [[4.68256853e+01, 8.18815132e+01, 1.13939960e+02,
          1.26987478e+02],
         [7.63317407e+01, 1.24581269e+02, 2.34609699e+02,
          3.36102146e+02],
         [1.59363421e+02, 2.65118856e+02, 3.60634819e+02,
          4.15322497e+02],
         [4.12361079e+01, 7.10671078e+01, 1.58178851e+02,
          1.72482790e+02],
         [1.64224862e+02, 2.45124690e+02, 3.40852076e+02,
...
          5.18831304e+04],
         [1.92291377e+04, 3.61907519e+04, 5.60000045e+04,
          6.76386420e+04],
         [2.09228807e+04, 3.98849001e+04, 6.24941363e+04,
          7.13389761e+04],
         [6.97500764e+03, 1.59926893e+04, 3.24376013e+04,
          3.68400700e+04],
         [1.85795430e+04, 3.72780205e+04, 5.68896111e+04,
          6.81998824e+04]],

        [[2.94848948e+04, 4.32491159e+04, 5.28359390e+04,
          5.57699690e+04],
         [5.85027659e+04, 7.38514543e+04, 1.48566228e+05,
          1.83077849e+05],
         [1.63404566e+05, 1.94409808e+05, 2.40059899e+05,
          2.61827164e+05],
         [2.46478906e+04, 3.33255109e+04, 5.83086695e+04,
          6.07213352e+04],
         [2.16270563e+05, 2.58588911e+05, 3.02525601e+05,
          3.43401686e+05]]]])
Coordinates:
  * quantile  (quantile) <U6 72B 'p5' 'actual' 'p95'
  * chem      (chem) <U3 60B 'TN' 'TP' 'Fe' 'dSi' 'TSS'
  * river     (river) <U6 120B 'Puelo' 'Yelcho' 'Palena' 'Cisnes' 'Aysen'
  * season    (season) <U6 96B 'Fall' 'Winter' 'Spring' 'Summer'

In [ ]:
annual_total_flux = cumboot_qs.isel(Date=-1).stack(x=["river", "quantile"]).to_pandas()
annual_total_flux.to_csv(csv_out_root+"annual_total_flux.csv")
annual_total_flux.to_excel(csv_out_root+"annual_total_flux.xlsx")


boot_qs_yield = boot_qs / xr.DataArray(areas, coords={"river": rivers})
cumboot_yield = cumboot_qs / xr.DataArray(areas, coords={"river": rivers})

#annual_sums_yield = boot_qs_yield.sum(dim="Date").stack(x=["river", "quantile"]).to_pandas()
annual_total_yield = cumboot_yield.isel(Date=-1).stack(x=["river", "quantile"]).to_pandas()
annual_total_yield.to_csv(csv_out_root+"annual_total_yield.csv")
annual_total_yield.to_excel(csv_out_root+"annual_total_yield.xlsx")

In [ ]:
monthly_totals = xr.concat([
    all_dailyboot
      .groupby("Date.month")
      .sum(dim="Date")
      .quantile([0.05, 0.95], dim="dim_1")
      .assign_coords(quantile=["p5", "p95"]),
    boot_qs
      .sel(quantile="actual")
      .groupby("Date.month")
      .sum(dim="Date")
    ], dim="quantile"
)
monthly_totals = monthly_totals.roll(month=-3, roll_coords=True)

seasonal_totals = xr.concat([
    all_dailyboot
      .groupby(season_codes)
      .sum(dim="Date")
      .quantile([0.05, 0.95], dim="dim_1")
      .assign_coords(quantile=["p5", "p95"], season=["Fall", "Winter", "Spring", "Summer"]),
    boot_qs.sel(quantile="actual")
      .groupby(season_codes)
      .sum(dim="Date")
      .assign_coords(season=["Fall", "Winter", "Spring", "Summer"])
    ], dim="quantile"
)

all_dailyboot_yield = all_dailyboot / xr.DataArray(areas, coords={"river": rivers})

monthly_totals_yield = xr.concat([
    all_dailyboot_yield
      .groupby("Date.month")
      .sum(dim="Date")
      .quantile([0.05, 0.95], dim="dim_1")
      .assign_coords(quantile=["p5", "p95"]),
    boot_qs_yield
      .sel(quantile="actual")
      .groupby("Date.month")
      .sum(dim="Date")
    ], dim="quantile"
)
monthly_totals_yield = monthly_totals_yield.roll(month=-3, roll_coords=True)

seasonal_totals_yield = xr.concat([
    all_dailyboot_yield
      .groupby(season_codes)
      .sum(dim="Date")
      .quantile([0.05, 0.95], dim="dim_1")
      .assign_coords(quantile=["p5", "p95"], season=["Fall", "Winter", "Spring", "Summer"]),
    boot_qs_yield.sel(quantile="actual")
      .groupby(season_codes)
      .sum(dim="Date")
      .assign_coords(season=["Fall", "Winter", "Spring", "Summer"])
    ], dim="quantile"
)

seasonal_totals_df = seasonal_totals.stack(x=["river", "chem"]).stack(y=["season", "quantile"]).to_pandas()
seasonal_totals_df.to_csv(csv_out_root+"seasonal_total_flux.csv")
seasonal_totals_df.to_excel(csv_out_root+"seasonal_total_flux.xlsx")

monthly_totals_df = monthly_totals.stack(x=["river", "chem"]).stack(y=["month", "quantile"]).to_pandas()
monthly_totals_df.to_csv(csv_out_root+"monthly_total_flux.csv")
monthly_totals_df.to_excel(csv_out_root+"monthly_total_flux.xlsx")

seasonal_totals_yield_df = seasonal_totals_yield.stack(x=["river", "chem"]).stack(y=["season", "quantile"]).to_pandas()
seasonal_totals_yield_df.to_csv(csv_out_root+"seasonal_total_yield.csv")
seasonal_totals_yield_df.to_excel(csv_out_root+"seasonal_total_yield.xlsx")

monthly_totals_yield_df = monthly_totals_yield.stack(x=["river", "chem"]).stack(y=["month", "quantile"]).to_pandas()
monthly_totals_yield_df.to_csv(csv_out_root+"monthly_total_yield.csv")
monthly_totals_yield_df.to_excel(csv_out_root+"monthly_total_yield.xlsx")

In [ ]:
import ipywidgets as widgets
tab = widgets.Tab()
tab.children = [widgets.Output() for i in range(6)]
tab.titles = ["Monthly Flux", "Monthly Yield", "Seasonal Flux", "Seasonal Yield", "Annual Flux", "Annual Yield"]
for (df, out) in zip(
    [monthly_totals_df, monthly_totals_yield_df, seasonal_totals_df, seasonal_totals_yield_df, annual_total_flux, annual_total_yield],
    tab.children):
  with out:
    display(df)

for i, t in enumerate(tab.titles):
  tab.set_title(i, t)
display(tab)

In [ ]:
# Truthing
assert (
    (cumboot_yield.sel(quantile="actual").isel(Date=-1) - boot_qs_yield.sel(quantile="actual").sum(dim="Date"))
    < 0.001).all()

In [ ]:
all_flows_smoothed

<xarray.DataArray (Date: 365, river: 5)> Size: 15kB
array([[348.66054937, 316.26591317, 589.02849962, 261.94995291,
        379.02254737],
       [367.61759316, 350.62333471, 610.00220153, 264.38107633,
        384.01054225],
       [393.30131775, 376.03690298, 647.88349596, 275.73600323,
        391.16335859],
       ...,
       [336.99154482, 219.00810776, 370.88589951, 221.80095488,
        557.26577966],
       [321.83178079, 228.98666373, 341.43702575, 214.71204828,
        532.17468292],
       [309.73335987, 231.03103336, 325.397283  , 210.45017229,
        506.30257102]])
Coordinates:
  * Date     (Date) datetime64[ns] 3kB 2022-04-01 2022-04-02 ... 2023-03-31
  * river    (river) object 40B 'Aysen' 'Cisnes' 'Palena' 'Puelo' 'Yelcho'

In [ ]:
def plot_river_background(ax, ylabel=True):
  ax1 = ax.twinx()
  # all_flows_smoothed.to_pandas().plot.area(color=streamflow_colors, alpha=0.3) #alternative
  ax1.stackplot(all_flows_smoothed["Date"],
                all_flows_smoothed.sel(river=rivers[::-1]).transpose(),
                colors=streamflow_colors[::-1],
                alpha=0.3)
  ax1.set_ylabel("Streamflow (m³/s)")

def pretty_dates(ax, date_index=None):
  ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
  ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y\n\n\n"))
  ax.tick_params(axis='x', labelrotation=45)

  if date_index is not None:
    ax.set_xlim(date_index[0],
                date_index[-1]+pd.Timedelta(days=1))

if add_fluxbiases:
  def format_fluxbiases(chem, rivers=rivers, round=4):
    return list(
        map(
            lambda t: (t[0] + ": " + t[1]),
            zip(
                rivers,
                all_errorstats.sel(chem=chem, dim_1="fluxBias.bias3")
                .sel(river=rivers)
                .round(decimals=round)
                .astype("str")
                .values
            )
        )
    )
else:
  def format_fluxbiases(chem, rivers=rivers, round=4):
    return rivers


river_cmap = mcolors.ListedColormap(rivercolors)

In [ ]:
# Matplotlib style stuff
# This is my replication of chatgpt's pretty plots, with slightly less shitty code
# Remove warnings for masked elements.
warnings.filterwarnings("ignore", message="^.*converting a masked element to nan.*$")
# thanks to OverLordGoldDragon https://stackoverflow.com/a/9134842

# Set professional style and font parameters.
sns.set_style("white")  # No gridlines
sns.set_context("talk", font_scale=1.2)  # Larger text
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = ['Times New Roman', 'Georgia', 'DejaVu Serif']
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 18
mpl.rcParams['xtick.labelsize'] = 16
mpl.rcParams['ytick.labelsize'] = 16
mpl.rcParams['legend.fontsize'] = 16
mpl.rcParams['lines.linewidth'] = 2


### test block
mpl.rcParams['axes.labelsize'] = 24    # Increase axis label size
mpl.rcParams['axes.titlesize'] = 24    # Increase title size (if needed)
mpl.rcParams['xtick.labelsize'] = 26   # Increase x tick label size
mpl.rcParams['ytick.labelsize'] = 26   # Increase y tick label size
mpl.rcParams['legend.fontsize'] = 28   # Increase legend font size
## test block
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams['font.serif'] = ['Arial']

In [ ]:
tall_plot_root = fig_out_root+"column/"

plt.ioff()
from matplotlib.gridspec import GridSpec
def tall_plot(da_flux, da_yield, ylabel1, ylabel2):

  fig, axes = plt.subplots(nrows=len(chemicals)+1,
                          ncols=2,
                          #figsize=(6*len(chemicals), 18))
                          figsize=(24,48),
                          gridspec_kw={'wspace': 0.5, 'hspace': 0.0},
  )#constrained_layout=True)
  fig.suptitle("", fontsize=16, y=0.91)
  #axes = axes.T
  #plt.subplots_adjust(left=0.15, wspace=0.27, hspace=0.32)
  #fig.suptitle("Cumulative Fluxes - Unscaled vs Scaled", fontsize=16, y=0.91)


  for chem, axes_row in zip(chemicals, axes):
    ax1, ax2 = axes_row
    plot_river_background(ax1)

    #ax.plot(total_flow, label="Total Flow", color="black") # add this to the function?

    ax1.set_prop_cycle(color=rivercolors)
    # plot confidence intervals
    for r, color in zip(rivers, rivercolors):
      ax1.fill_between(date_idx,
                      da_flux.sel(chem=chem, river=r, quantile="p5"),
                      da_flux.sel(chem=chem, river=r, quantile="p95"),
                      #color=color,
                      alpha=0.3
                      )
    # plot it
    #cumboot_qs.sel(chem=c, quantile=0.5).plot.line(x="Date"))
    ax1.plot(date_idx,
            da_flux.sel(chem=chem, quantile="actual").transpose(),
            label=format_fluxbiases(chem, rivers))

    # plot sample points
    for r, color in zip(rivers, point_colors):
      da_flux.sel(chem=chem, river=r, quantile="actual").where(
                    all_samples.sel(chem=chem, river=r).notnull()
                ).plot.scatter(
                    x="Date",
                    color=color,
                    ax=ax1,
                    add_colorbar=False,
                    add_legend=False)

    ax1.set_title(chem, y=0.8)
    ax1.set_ylabel(ylabel1)
    #ax1.legend(loc="upper left")
    #pretty_dates(ax1, date_index=water_year)
    ax1.get_xaxis().set_visible(False)


    # Now plot for the 2nd column
    plot_river_background(ax2)

    ax2.set_prop_cycle(color=rivercolors)
    # plot confidence intervals
    for r, color in zip(rivers, rivercolors):
      ax2.fill_between(date_idx,
                      da_yield.sel(chem=chem, river=r, quantile="p5"),
                      da_yield.sel(chem=chem, river=r, quantile="p95"),
                      #color=color,
                      alpha=0.3
                      )

    ax2.plot(date_idx, da_yield.sel(chem=chem, quantile="actual").transpose(), label=rivers)

    # plot sample points
    for r, color in zip(rivers, point_colors):
      da_yield.sel(chem=chem, river=r, quantile="actual").where(
                    all_samples.sel(chem=chem, river=r).notnull()
                ).plot.scatter(
                    x="Date",
                    color=color,
                    ax=ax2,
                    add_colorbar=False,
                    add_legend=False)

    ax2.set_title(chem, y=0.8)
    ax2.set_ylabel(ylabel2)
    #ax2.legend(loc="upper left")
    #pretty_dates(ax2, date_index=water_year)
    ax2.get_xaxis().set_visible(False)


  ax1.get_xaxis().set_visible(True)
  ax2.get_xaxis().set_visible(True)
  pretty_dates(ax1, date_index=date_idx)
  pretty_dates(ax2, date_index=date_idx)
  for ax in axes.flat:
    ax.set_xlim(ax2.get_xlim())

  # Use the last row for legend
  gs = GridSpec(*axes.shape, figure=fig)
  for ax in axes[-1]: ax.remove()
  leg_ax = fig.add_subplot(gs[-1, :], )
  leg_ax.legend(*ax1.get_legend_handles_labels(), loc="center")
  leg_ax.axis("off")

  fig.align_ylabels()
  axes[0,0].set_title("Flux\n\n\n\n"+chemicals[0], y=0.8)
  axes[0,1].set_title("Yield\n\n\n\n"+chemicals[0], y=0.8)

  return fig


tab = widgets.Tab()
tab.children = [widgets.Output() for i in range(2)]
child1, child2 = tab.children

fig1 = tall_plot(cumboot_qs, cumboot_yield, "Cumulative Flux\n(metric tonnes)", "Cumulative Yield\n(metric tonnes / sq. km)")
fig1.savefig(tall_plot_root+"cumulative_flux_and_yield_all.jpg")
with child1:
  display(fig1)
tab.set_title(0, "Cumulative")

fig2 = tall_plot(boot_qs, boot_qs_yield, "Daily Flux\n(metric tonnes)", "Daily Yield\n(metric tonnes / sq. km)")
fig2.savefig(tall_plot_root+"daily_flux_and_yield_all.jpg")
with child2:
  display(fig2)
tab.set_title(1, "Daily")

display(tab)
plt.ion()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Define the custom ordering of chemicals (5 items).
#chemicals_order = ["dSi", "TSS", "Fe", "TN", "TP"]

def plot5chem(da1, chemicals_order, ylabel):
  # Create a 3x2 grid of subplots.
  fig, axes = plt.subplots(nrows=3, ncols=2,
                          figsize=(24,20),
                          gridspec_kw={'wspace': 0.55, 'hspace': 0.03},
                          tight_layout=True)
  fig.suptitle("", fontsize=20, y=0.91)

  for (chem, ax) in zip(chemicals_order, axes.flat):
    # Select the slice of cumboot_scaled for the specifed chemical
    da = da1.sel(chem=chem)

    # Plot background (this creates the secondary axis, etc.).
    plot_river_background(ax, ylabel=False)
    # (Optional: If you want to rely on automatic color cycling, you can leave this in.
    # Here we explicitly override colors for each element below.)
    # ax.set_prop_cycle(color=rivercolors)

    # Plot the confidence intervals
    for r, color in zip(rivers, rivercolors):
      ax.fill_between(date_idx,
                      da.sel(river=r, quantile="p5"),
                      da.sel(river=r, quantile="p95"),
                      alpha=0.3,
                      color=color)

    # Plot the "actual" flux
    ax.set_prop_cycle(color=rivercolors)
    lines = da.sel(quantile="actual").plot.line(ax=ax, x="Date", add_legend=False) # Need a legend for the final one

    # Plot the sampled points. Note: The y-values of the sampled points is not true for flux plots
    for r, color in zip(rivers, point_colors):
      da.sel(river=r, quantile="actual").where(
                    all_samples.sel(chem=chem, river=r).notnull()
                ).plot.scatter(
                    x="Date",
                    color=color,
                    ax=ax,
                    add_colorbar=False,
                    add_legend=False)

    # #BRIGHTER POINTS CODE
    # #darker-shade points with white borders
    # for idx, r_col in enumerate(rivers):
    #     da_actual = da.sel(river=r_col, quantile="actual")
    #     mask = all_samples.sel(chem=chem, river=r_col).notnull()
    #     pts = da_actual.where(mask).dropna(dim="Date")
    #     ax.scatter(
    #         pts['Date'].values,
    #         pts.values,
    #         marker='o',
    #         facecolors=point_colors[idx],  # darker fill
    #         edgecolors='white',            # white border
    #         linewidths=0.8,
    #         s=80,
    #         zorder=lines[0].get_zorder() + 1
    #     )


    # Title
    ax.set_title(chem, y=0.8, fontsize=30)

    # Make the x ticks pretty
    ax.set_xlabel("")
    ax.set_xlim(date_idx[0], date_idx[-1]+pd.Timedelta(days=1)) # So that theres a final xtick after march 31 2023
    ax.set_xticklabels([])
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.set_ylabel(ylabel, fontsize=26)
    #pretty_dates(ax, date_index=water_year)

  # Create the legend for the bottom right ax
  axes[-1,-1].legend(lines, rivers, loc="lower center")
  axes[-1,-1].axis("off")

  # Put the date axis on the bottom
  pretty_dates(axes[2,0], date_index=water_year)
  pretty_dates(axes[1,1], date_index=water_year)

  # Remove that nasty space and align the ylabels
  plt.subplots_adjust(hspace=0.0)
  fig.align_ylabels()

  return fig

plt.ioff()
tab = widgets.Tab()
tab.children = [widgets.Output() for i in range(5)]
child1, child2, child3, child4, child5 = tab.children

fig1 = plot5chem(cumboot_yield, ["dSi", "TSS", "Fe", "TN", "TP"], "Cumulative Yield\n(tonnes / km²)")
with child1:
  display(fig1)
tab.set_title(0, "Cumulative Yield")
fig1.savefig(tall_plot_root+"cumulative_yield_all.png")

fig2 = plot5chem(boot_qs_yield, ["dSi", "TSS", "Fe", "TN", "TP"], "Daily Yield\n(tonnes / km²)")
with child2:
  display(fig2)
tab.set_title(1, "Daily Yield")
fig2.savefig(tall_plot_root+"daily_yield_all.png")

fig3 = plot5chem(cumboot_qs, ["dSi", "TSS", "Fe", "TN", "TP"], "Cumulative Flux\n(tonnes)")
with child3:
  display(fig3)
tab.set_title(2, "Cumulative Flux")
fig3.savefig(tall_plot_root+"cumulative_flux_all.png")

fig4 = plot5chem(boot_qs, ["dSi", "TSS", "Fe", "TN", "TP"], "Daily Flux\n(tonnes)")
with child4:
  display(fig4)
tab.set_title(3, "Daily Flux")
fig4.savefig(tall_plot_root+"daily_flux_all.png")

conc_qs = xr.concat(
    [
            all_dayPctConc.sel(dim_1=["p5", "p95"]).rename({"dim_1": "quantile"}),
            all_daily_results.sel(dim_1="GenConc").rename({"dim_1": "quantile"}).assign_coords({"quantile": "actual"})
    ],
    dim="quantile"
    )
converted_to_ug= False

fig5 = plot5chem(conc_qs, ["dSi", "TSS", "Fe", "TN", "TP"], "Concentration\n(mg/L)")
with child5:
  display(fig5)
tab.set_title(4, "Concentration")
fig5.savefig(tall_plot_root+"conc_all.jpeg")

display(tab)

plt.ion()


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
stackwide_fig_root = fig_out_root+"stackwide/"
# if not os.path.exists(fig_out_root):
#   os.makedirs(fig_out_root)

# Define the custom ordering of chemicals (5 items).
#chemicals_order = ["dSi", "TSS", "Fe", "TN", "TP"]

def plot5chem_stackwide(da1, chemicals_order, ylabel):
  # Create a 3x2 grid of subplots.
  fig, axes = plt.subplots(nrows=len(chemicals_order),
                          figsize=(18, 5*len(rivers)),
                          gridspec_kw={'hspace': 0.03},
                          tight_layout=True)
  #fig.suptitle("", fontsize=20, y=0.91)

  for (chem, ax) in zip(chemicals_order, axes.flat):
    # Select the slice of cumboot_scaled for the specifed chemical
    da = da1.sel(chem=chem)

    # Plot background (this creates the secondary axis, etc.).
    plot_river_background(ax, ylabel=False)
    # (Optional: If you want to rely on automatic color cycling, you can leave this in.
    # Here we explicitly override colors for each element below.)
    # ax.set_prop_cycle(color=rivercolors)

    # Plot the confidence intervals
    for r, color in zip(rivers, rivercolors):
      ax.fill_between(date_idx,
                      da.sel(river=r, quantile="p5"),
                      da.sel(river=r, quantile="p95"),
                      alpha=0.3,
                      color=color)

    # Plot the "actual" flux
    ax.set_prop_cycle(color=rivercolors)
    lines = da.sel(quantile="actual").plot.line(ax=ax, x="Date", add_legend=False) # Need a legend for the final one

    # Plot the sampled points. Note: The y-values of the sampled points is not true for flux plots
    for r, color in zip(rivers, point_colors):
      da.sel(river=r, quantile="actual").where(
                    all_samples.sel(chem=chem, river=r).notnull()
                ).plot.scatter(
                    x="Date",
                    color=color,
                    ax=ax,
                    add_colorbar=False,
                    add_legend=False)
    # Title
    ax.set_title(chem, y=0.8, fontsize=30)

    # Make the x ticks pretty
    ax.set_xlabel("")
    ax.set_xlim(date_idx[0], date_idx[-1]+pd.Timedelta(days=1)) # So that theres a final xtick after march 31 2023
    ax.set_xticklabels([])
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.set_ylabel(ylabel, fontsize=26)
    #pretty_dates(ax, date_index=water_year)

  # Create the legend for the bottom right ax
  #axes[-1].legend(lines, rivers, loc="lower center")

  # Put the date axis on the bottom
  pretty_dates(axes[-1], date_index=water_year)

  # Remove that nasty space and align the ylabels
  plt.subplots_adjust(hspace=0.0)
  fig.align_ylabels()

  return fig

plt.ioff()
tab = widgets.Tab()
tab.children = [widgets.Output() for i in range(5)]
child1, child2, child3, child4, child5 = tab.children

#temporarily converting to kg/km2
cumboot_yield.loc[{"chem": "TN"}] *= 1000
cumboot_yield.loc[{"chem": "TP"}] *= 1000
cumboot_yield.loc[{"chem": "Fe"}] *= 1000

fig1 = plot5chem_stackwide(cumboot_yield, chemicals, "Cumulative Yield\n(tonnes / km²)")

for ax in fig1.get_axes():
  if ax.title.get_text() in {"TN", "TP", "Fe"}:
    ax.set_ylabel("Cumulative Yield\n(kg / km²)")

with child1:
  display(fig1)
tab.set_title(0, "Cumulative Yield")
fig1.savefig(stackwide_fig_root+"cumulative_yield_sw.png", dpi=800, bbox_inches="tight")

#temporarily converting to kg/km2
cumboot_yield.loc[{"chem": "TN"}] /= 1000
cumboot_yield.loc[{"chem": "TP"}] /= 1000
cumboot_yield.loc[{"chem": "Fe"}] /= 1000

fig2 = plot5chem_stackwide(boot_qs_yield, chemicals, "Daily Yield\n(tonnes / km²)")
with child2:
  display(fig2)
tab.set_title(1, "Daily Yield")
fig2.savefig(stackwide_fig_root+"daily_yield_sw.png", dpi=300, bbox_inches="tight")

fig3 = plot5chem_stackwide(cumboot_qs, chemicals, "Cumulative Flux\n(tonnes)")
with child3:
  display(fig3)
tab.set_title(2, "Cumulative Flux")
fig3.savefig(stackwide_fig_root+"cumulative_flux_sw.png", dpi=300, bbox_inches="tight")

fig4 = plot5chem_stackwide(boot_qs, chemicals, "Daily Flux\n(tonnes)")
with child4:
  display(fig4)
tab.set_title(3, "Daily Flux")
fig4.savefig(stackwide_fig_root+"daily_flux_sw.png", dpi=300, bbox_inches="tight")

if not converted_to_ug:
    converted_to_ug = True
    # Convert TN, TP, and Fe to ug/L
    conc_qs.loc[{"chem": "TN"}] *= 1000
    conc_qs.loc[{"chem": "TP"}] *= 1000
    conc_qs.loc[{"chem": "Fe"}] *= 1000

fig5 = plot5chem_stackwide(conc_qs, chemicals, "Concentration\n(mg/L)")
# Need to go in there and change the ylabels too
for ax in fig5.get_axes():
  if ax.title.get_text() in {"TN", "TP", "Fe"}:
    ax.set_ylabel("Concentration\n(ug/L)")

with child5:
  display(fig5)
tab.set_title(4, "Concentration")
fig5.savefig(stackwide_fig_root+"concentration_annual.png", dpi=300, bbox_inches="tight")

# Also do it with a log scale
for ax in fig5.get_axes():
  if "Concentration" in ax.get_ylabel():
    ax.set_yscale("log")
    ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10, subs=(0.5,1,2)))
    ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.yaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
  if ax.title.get_text() == "TN":
    ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10, subs=(2, 5)))
  if ax.title.get_text() == "TP":
    ax.set_ylim(ax.get_ylim()[0], ax.get_ylim()[1]*2)
  if ax.title.get_text() == "TSS":
    ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10, subs=(0.02, 0.05)))
  if ax.title.get_text() == "Fe":
    ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10, subs=(0.05, 0.2)))

fig5.savefig(stackwide_fig_root+"concentration_annual_log.png", dpi=300, bbox_inches="tight")

for ax in fig5.get_axes():
    ax.set_title("")
fig5.savefig(stackwide_fig_root+"concentration_annual_log_nolabel.png", dpi=800, bbox_inches="tight")


display(tab)

plt.ion()

In [ ]:
rivers.reverse()

In [ ]:
# for da, tipo, unit in zip(
#     [boot_qs,          ],#boot_qs_scaled, cumboot_qs,            cumboot_scaled],
#     ["unscaled daily", ],#"scaled daily", "unscaled cumulative", "scaled cumulative"],
#     ["metric tonnes",  ],#"mt/km2",       "mt",                  "mt/km2"]
#     ):
#     for chem in chemicals[:1]:
#       fig, axes = plt.subplots(nrows=len(rivers),
#                                figsize=(12, 6*len(rivers)),
#                                gridspec_kw={"hspace": 0.05})
#       fig.suptitle(f"{chem} flux {tipo}", y=0.9)
#       for riv, clr, ax in zip(rivers, rivercolors, axes):
#         da2 = da.sel(chem=chem, river=riv)
#         plot_river_background(ax)
#         ax.plot(date_idx, da2.sel(quantile="actual"), color=clr)
#         ax.fill_between(date_idx,
#                         da2.sel(quantile="p5"),
#                         da2.sel(quantile="p95"),
#                         color=clr,
#                         alpha=0.3)
#         # add points
#         ax.set_title(riv, loc="left", y=0.9)
#         ax.set_ylabel(f"Flux ({unit})")
#       for ax in axes[:-1]:
#         ax.xaxis.set_visible(False)
#       #axes[-1].set_xlabel("Date")
#       pretty_dates(axes[-1], date_index=date_idx)
#       fig.align_ylabels()
#       tipo = tipo.replace(" ", "_")
#       fig.savefig(stackwide_fig_root+f"{chem}_stackwide_{tipo}.png", dpi=300, bbox_inches="tight")


In [ ]:
barplot_fig_root = fig_out_root+"barplots/"
seasonal_fig_root = barplot_fig_root+"seasonal/"
monthly_fig_root = barplot_fig_root+"monthly/"

In [ ]:
# # Shift the monthly totals so it starts in April (the start of the water year)
# if monthly_totals["month"][0] == 1:
#   monthly_totals = monthly_totals.roll(month=-3, roll_coords=True)

# Plot monthly boxplots

def plot_totals(river, chem, total_da):
    da = total_da.sel(river=river, chem=chem)
    fig, ax2 = plt.subplots(figsize=(16, 8))

    #plot_river_background(ax)
    ax2.fill_between(date_idx,
                     all_flows_smoothed.sel(river=river),
                     color="lightgray", edgecolor="black",
                     alpha=0.3)
    ax2.set_ymargin(0.0) # Sets both top and bottom margin, unaddressed issue
    ax2.set_ylim(0.0, ax2.get_ylim()[1]*1.1) # to fix top margin
    ax2.set_ylabel("Streamflow (m³/s)")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.xaxis.set_visible(False)

    ax = fig.add_subplot(111, frameon=False)
    idx = da["month"] if "month" in da.indexes else da["season"]
    ax.bar(idx.astype("str"),
        height=da.sel(quantile="actual").values,
        yerr=da.sel(quantile=["p5", "actual", "p95"]).diff(dim="quantile"),
        #color=river_cmap,
        alpha=0.7,
        capsize=5)
    #ax.set_xticklabels(idx.values)
    ax.set_ylabel("Flux (metric tonnes)")
    ax.set_title(f"{river} {chem}")
    return fig
plot_totals("Aysen", "dSi", monthly_totals)
for r in rivers:
  for c in chemicals:
    fig = plot_totals(r, c, monthly_totals)
    #fig.savefig("{monthly_fig_root}{r}_{c}_monthly_total_bar.jpg")
    # Sage temporary savinf workaround
    fig.savefig(monthly_fig_root+"monthly_total_bar.jpg")


####### Section 2

for r in rivers:
  for c in chemicals:
    fig = plot_totals(r, c, seasonal_totals)
    #fig.savefig(f"{seasonal_fig_root}{r}_{c}_seasonal_total_bar.jpg")
    #Sage temp saving workaround
    fig.savefig(seasonal_fig_root+"seasonal_total_bar.jpg")



for da, bar_x, fr in zip([monthly_totals,   seasonal_totals],
                               ["month",          "season"],
                               [monthly_fig_root, seasonal_fig_root]):
  for chem in chemicals:
    fig, axes = plt.subplots(nrows=len(rivers),
                             figsize=(20, 6*len(rivers)),
                             sharex=True,
                             gridspec_kw={"hspace": 0.03})
    for r, ax2 in zip(rivers, axes):
      # plot river background
      ax2.fill_between(date_idx,
                      all_flows_smoothed.sel(river=r),
                      color="lightgray", edgecolor="black",
                      alpha=0.3)
      ax2.set_ymargin(0.0) # Sets both top and bottom margin, unaddressed issue
      ax2.set_ylim(0.0, ax2.get_ylim()[1]*1.1) # to fix top margin
      ax2.set_ylabel("Streamflow (m³/s)")
      ax2.yaxis.tick_right()
      ax2.yaxis.set_label_position("right")
      ax2.xaxis.set_visible(False)

      ax = fig.add_axes(ax2.get_position(), frameon=False)
      ax.bar(da[bar_x].astype("str"),
          da.sel(chem=chem, river=r, quantile="actual"),
          yerr=da.sel(chem=chem, river=r, quantile=["p5", "actual", "p95"]).diff(dim="quantile"),
          #color=river_cmap,
          alpha=0.7,
          capsize=5)
          #tick_label="")
      ax.xaxis.set_visible(False)
      ax.set_ylabel(f"{r}\nFlux (metric tonnes)")

    ax.xaxis.set_visible(True) # last axes
    ax.set_xlabel(bar_x)
    #axes[0].set_title(chem)
    fig.align_ylabels()
    fig.suptitle(f"{chem} Flux by Month", y=0.9)
    #fig.savefig(fr+f"{chem}_5_{bar_x}_bar.jpg", bbox_inches="tight") # THANK YOUU https://stackoverflow.com/a/45239920
    #Sage saving workaround
    fig.savefig(fr+f"{chem}_5_{bar_x}_bar.jpg", bbox_inches="tight")



In [ ]:
#### Sage alternative from ChatGPT to avoid error on sage laptop, April 26 - uses rolling avg streamflow

import numpy as np

months = [
    "Apr 2022", "May 2022", "Jun 2022", "Jul 2022",
    "Aug 2022", "Sep 2022", "Oct 2022", "Nov 2022",
    "Dec 2022", "Jan 2023", "Feb 2023", "Mar 2023"
]

# 1) compute the integer percentages
p = (
    (monthly_totals / monthly_totals.sum(dim="river") * 100)
    .sel(quantile="actual")
    .round(0)
    .astype(int)
)

# 2) cast to Python objects so that '+' does string concat, not numeric ufunc
p_obj = p.astype(str).astype(object)

# 3) add the percent sign
percentages_inline = p_obj + '%'

# 4) build your mask of “small” months
mask = (
    (monthly_totals / monthly_totals.sum(dim="river").max(dim="month"))
    .sel(quantile="actual")
    < 0.04
)

# 5) split into above‐threshold and inline tables
percentages_above  = percentages_inline.where(mask)
percentages_inline = percentages_inline.where(~mask, "")

# 6) compute a 30-day centered rolling average over the 'Date' dimension with no NaNs at the edges
total_flow_np = (
    all_flows
    .sum(dim="river")
    .rolling(Date=30, center=True, min_periods=1)
    .mean()
    .values
)


In [ ]:
#### Sage temporary saving solution, delete later once errors are resolved

out_dir_temp = '/Users/sagefox/My Drive (sagefox@uw.edu)/Research/Python_Code/Paper/figures/chemistry/flux/WRTDS_flux/monthly/presentation/'

In [ ]:
####### SAGE MODIFIED STACKED BAR PLOTTING CODE, temporary for presentation - merge with nicks plotting code later
stacked_monthly_fig_root = barplot_fig_root+"stacked_monthly/"

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

####### Sage modification for presentation - temporary

fig, axes = plt.subplots(
    nrows=len(chemicals),
    figsize=(16, 8*len(chemicals)),
    tight_layout=True
)

for chem, ax, i in zip(chemicals, axes, range(len(chemicals))):
    # ——— Stacked bar plot ———
    ax.set_prop_cycle(color=rivercolors)

    # 1) Compute the per-month sums
    actual_sum = monthly_totals.sel(quantile="actual", chem=chem).sum("river")
    bottoms    = actual_sum.values.copy()

    # 2) Stack each river’s bar
    for riv in rivers:
        heights = monthly_totals.sel(
            quantile="actual", river=riv, chem=chem
        ).values
        bottoms -= heights
        ax.bar(
            months,
            heights,
            bottom=bottoms,
            label=riv,
            alpha=0.7
        )

    # ——— Cumulative error bars for the sum ———
    lower_sum = monthly_totals.sel(quantile="p5", chem=chem).sum("river")
    upper_sum = monthly_totals.sel(quantile="p95", chem=chem).sum("river")
    lower_err = (actual_sum - lower_sum).values
    upper_err = (upper_sum - actual_sum).values
    yerr      = np.vstack([lower_err, upper_err])

    ax.errorbar(
        np.arange(len(months)),
        actual_sum.values,
        yerr=yerr,
        fmt='none',
        ecolor='black',
        capsize=5,
        zorder=10
    )

    # ——— Formatting ———
    ax.set_xticklabels(
        months,
        fontdict={
            "rotation": 45,
            "ha": "right",
            "rotation_mode": "anchor"
        }
    )
    #ax.legend()
    ax.set_ylabel("Flux (metric tonnes)")
    ax.set_title(chem)

    # ——— Overlay streamflow ———
    ax2 = ax.twinx()
    ax2.fill_between(
        np.linspace(-0.4, 11.38, 365),
        total_flow_np,
        color="lightgray",
        edgecolor="black",
        alpha=0.3
    )
    ax2.set_ylim(0.0, ax2.get_ylim()[1])
    ax2.set_ylabel("Streamflow (m³/s)")

    # ensure the bars stay in front
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

# ——— Save individual panels ———
for i, chem in enumerate(chemicals):
    fig.savefig(
        stacked_monthly_fig_root+f"{chem}_stacked_monthly_flux.png",
        bbox_inches=mpl.transforms.Bbox([[0, i*8], [16, (i+1)*8]])
    )

fig.align_ylabels()

# hide x-ticks & legends on all but the last
# for ax in axes[:-1]:
#     ax.get_legend().remove()
#     ax.set_xticks([])

# ——— Save full stacked overview ———
fig.savefig(stacked_monthly_fig_root+"all_constituents_stacked_monthly_flux.png")


############ SCALED BELOW THIS (ABOVE IS UNSCALED


fig, axes = plt.subplots(
    nrows=len(chemicals),
    figsize=(16, 8*len(chemicals)),
    tight_layout=True
)

for chem, ax, i in zip(chemicals, axes, range(len(chemicals))):
    # ——— Stacked bar plot ———
    ax.set_prop_cycle(color=rivercolors)

    # 1) Compute the per-month sums
    actual_sum = monthly_totals_yield.sel(quantile="actual", chem=chem).sum("river")
    bottoms    = actual_sum.values.copy()

    # 2) Stack each river’s bar
    for riv in rivers:
        heights = monthly_totals_yield.sel(
            quantile="actual", river=riv, chem=chem
        ).values
        bottoms -= heights
        ax.bar(
            months,
            heights,
            bottom=bottoms,
            label=riv,
            alpha=0.7
        )

    # ——— Cumulative error bars for the sum ———
    lower_sum = monthly_totals_yield.sel(quantile="p5", chem=chem).sum("river")
    upper_sum = monthly_totals_yield.sel(quantile="p95", chem=chem).sum("river")
    lower_err = (actual_sum - lower_sum).values
    upper_err = (upper_sum - actual_sum).values
    yerr      = np.vstack([lower_err, upper_err])

    ax.errorbar(
        np.arange(len(months)),
        actual_sum.values,
        yerr=yerr,
        fmt='none',
        ecolor='black',
        capsize=5,
        zorder=10
    )

    # ——— Formatting ———
    ax.set_xticklabels(
        months,
        fontdict={
            "rotation": 45,
            "ha": "right",
            "rotation_mode": "anchor"
        }
    )
    #ax.legend()
    ax.set_ylabel("Yield (tonnes/km²)")
    ax.set_title(chem)

    # ——— Overlay streamflow ———
    ax2 = ax.twinx()
    ax2.fill_between(
        np.linspace(-0.4, 11.38, 365),
        total_flow_np,
        color="lightgray",
        edgecolor="black",
        alpha=0.3
    )
    ax2.set_ylim(0.0, ax2.get_ylim()[1])
    ax2.set_ylabel("Streamflow (m³/s)")

    # ensure the bars stay in front
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

# ——— Save individual panels ———
for i, chem in enumerate(chemicals):
    fig.savefig(
        stacked_monthly_fig_root+f"{chem}_stacked_monthly_yield.png",
        bbox_inches=mpl.transforms.Bbox([[0, i*8], [16, (i+1)*8]])
    )

fig.align_ylabels()

# hide x-ticks & legends on all but the last
# for ax in axes[:-1]:
#     ax.get_legend().remove()
#     ax.set_xticks([])

# ——— Save full stacked overview ———
fig.savefig(stacked_monthly_fig_root+"all_constituents_stacked_monthly_yield.png")


In [ ]:
# ####### HORIOZONTAL BAR BASE PLOTTING CODE FROM NICK, WITH SMALL MODIFICATIONS FROM SAGE

# # Copied from a geospatial workboo, label positions for reference
# # ax[0].text(.6, .86, "Puelo", color='white', size=14, horizontalalignment='left', verticalalignment='bottom', transform=ax[0].transAxes)
# # ax[0].text(.6, .65, "Palena", color='white', size=14, horizontalalignment='left', verticalalignment='bottom', transform=ax[0].transAxes)
# # ax[0].text(.5, .45, "Yelcho", color='white', size=14, horizontalalignment='left', verticalalignment='bottom', transform=ax[0].transAxes)
# # ax[0].text(.6, .3, "Cisnes", color='white', size=14, horizontalalignment='left', verticalalignment='bottom', transform=ax[0].transAxes)
# # ax[0].text(.32, .14, "Aysen", color='white', size=14, horizontalalignment='left', verticalalignment='bottom', transform=ax[0].transAxes)

# y = [0.86, 0.65, 0.45, 0.3, 0.14]


# # horizontal bar plots
# horizontal_fig_root = barplot_fig_root+"horizontal_annual/"
# fig, axes = plt.subplots(ncols=len(chemicals), figsize=(8*len(chemicals), 4*len(rivers)), gridspec_kw={"wspace": 0.0})
# for (chem, ax) in zip(chemicals, axes):
#   annuals = annual_total_yield.loc[chem].unstack("river")
#   p = ax.barh(y,
#           annuals.loc["actual"],
#           height=0.1,
#           xerr=annuals.diff().drop("p5"),
#           color=rivercolors,
#           #tick_label=rivers
#           )
#   ax.bar_label(p,
#                labels=[np.format_float_scientific(t, precision=3) for t in annuals.loc["actual"].values],
#                label_type="center")
#   ax.invert_xaxis()
#   ax.set_title(f"{chem} horizontal")
#   ax.set_xlabel("Annual Yield (metric tonnes / km²)")




# fig, axes = plt.subplots(ncols=len(chemicals), figsize=(8*len(chemicals), 4*len(rivers)), gridspec_kw={"wspace": 0.0})
# for (chem, ax) in zip(chemicals, axes):
#   annuals = annual_total_flux.loc[chem].unstack("river")
#   p = ax.barh(y,
#           annuals.loc["actual"],
#           height=0.1,
#           xerr=annuals.diff().drop("p5"),
#           color=rivercolors,
#           #tick_label=rivers
#           )
#   ax.bar_label(p,
#                labels=[np.format_float_scientific(t, precision=3) for t in annuals.loc["actual"].values],
#                label_type="center")
#   ax.invert_xaxis()
#   ax.set_title(f"{chem} horizontal")
#   ax.set_xlabel("Annual Flux (metric tonnes)")
# #fig.savefig(horizontal_fig_root+f"all_horizontal_annual_bar.jpg")
# fig.savefig("../../../figures/chemistry/flux/WRTDS_flux/barplots/horizontal_annual/testing.jpg")

In [ ]:
###### SAGE EDITED HORIZONTAL BARPLOTS FOR PRESENTATION ONLY - KEEP THIS AND ABOVE

import numpy as np
import matplotlib.pyplot as plt

# your existing definitions
# chemicals, rivers, annual_totals, rivercolors, barplot_fig_root

# y‐positions for each river
y = [0.86, 0.65, 0.45, 0.3, 0.14]

# horizontal bar plots
horizontal_fig_root = barplot_fig_root + "horizontal_annual/"

for (df, fname, xlabel) in zip([annual_total_flux, annual_total_yield],
                               ["all_horizontal_annual_flux.png", "all_horizontal_annual_yield.png"],
                               ["Annual Flux (metric tonnes)", "Annual Yield (metric tonnes / km²)"]):
  fig, axes = plt.subplots(
      ncols=len(chemicals),
      figsize=(6.5 * len(chemicals), 4 * len(rivers)),
      gridspec_kw={"wspace": 0.0}
  )

  for chem, ax in zip(chemicals, axes):
      annuals = df.loc[chem].unstack("river")

      # 1) Draw the bars with caps on the error bars
      p = ax.barh(
          y,
          annuals.loc["actual"],
          height=0.08,                    # narrower bars
          xerr=annuals.diff().drop("p5"),
          capsize=3,                      # little T-caps
          error_kw={"elinewidth": 1},     # thinner error‐bar lines
          color=rivercolors,
          edgecolor="black",
      )

      # 2) Only label the 3-digit mantissa (drop the "e+XX")
      # mantissas = [
      #     np.format_float_scientific(val, precision=2).split("e")[0]
      #     for val in annuals.loc["actual"].values
      # ]
      mantissas = (
          annuals.loc["actual"] / (10 **
                                   int(np.log10(annuals.min(axis=None)))
                                   )
      ).round(2)
      ax.bar_label(p, labels=mantissas, label_type="center", fontsize=32)

      # 3) Clean up stray numbers by hiding all ticks
      ax.set_xticks([])
      ax.set_yticks([])

      # 4) Finish styling
      ax.invert_xaxis()
      ax.set_title(f"{chem} horizontal")
      ax.set_xlabel(xlabel)

      plt.tight_layout()
      # you can save it as usual:
      # fig.savefig(horizontal_fig_root + "all_horizontal_annual_bar.png", dpi=300, bbox_inches="tight")
      plt.show()


      script_dir = os.getcwd()   # assumes you launched Python from your script’s folder
      #outpath = os.path.join(script_dir, fname)
      outpath = horizontal_fig_root+fname
      fig.savefig(outpath, dpi=300, bbox_inches="tight", transparent=True)

  #fig.savefig("../../../figures/chemistry/flux/WRTDS_flux/barplots/horizontal_annual/testing.jpg")

In [ ]:
# ###This is a chat modification of above plotting code - Keep

# # Remove warnings for masked elements.
# warnings.filterwarnings("ignore", message="^.*converting a masked element to nan.*$")

# # # Set professional style and font parameters.
# # sns.set_style("white")  # No gridlines
# # sns.set_context("talk", font_scale=1.2)  # Larger text
# # mpl.rcParams['font.family'] = 'serif'
# # mpl.rcParams['font.serif'] = ['Times New Roman', 'Georgia', 'DejaVu Serif']
# # mpl.rcParams['axes.titlesize'] = 20
# # mpl.rcParams['axes.labelsize'] = 18
# # mpl.rcParams['xtick.labelsize'] = 16
# # mpl.rcParams['ytick.labelsize'] = 16
# # mpl.rcParams['legend.fontsize'] = 16
# # mpl.rcParams['lines.linewidth'] = 2

# # Use a muted color palette for the river lines.
# rivercolors = sns.color_palette("muted", n_colors=len(rivers))

# # # Prepare scaled cumulative flux.
# # cumboot_scaled = cumboot_qs / (xr.DataArray(areas, coords={"river": rivers}))
# # cumboot_scaled.attrs["units"] = "mt/km^2"
# # cumboot_qs.attrs["units"] = "mt"

# # Define the custom ordering of chemicals (5 items).
# chemicals_order = ["dSi", "TSS", "Fe", "TN", "TP"]

# ### test block
# # mpl.rcParams['axes.labelsize'] = 24    # Increase axis label size
# # mpl.rcParams['axes.titlesize'] = 24    # Increase title size (if needed)
# # mpl.rcParams['xtick.labelsize'] = 24   # Increase x tick label size
# # mpl.rcParams['ytick.labelsize'] = 24   # Increase y tick label size
# # mpl.rcParams['legend.fontsize'] = 28   # Increase legend font size
# ## test block

# # Create a 3x2 grid of subplots.
# fig, axes = plt.subplots(nrows=3, ncols=2,
#                          figsize=(24,20),
#                          gridspec_kw={'wspace': 0.09, 'hspace': 0.03},
#                          constrained_layout=True)
# fig.suptitle("", fontsize=20, y=0.91)

# # Loop over the chemicals_order and corresponding axes.
# # (Only the first 5 axes will be used for plotting.)
# for c, ax in zip(chemicals_order, axes.flat):
#     # Plot background (this creates the secondary axis, etc.).
#     plot_river_background(ax)
#     # (Optional: If you want to rely on automatic color cycling, you can leave this in.
#     # Here we explicitly override colors for each element below.)
#     # ax.set_prop_cycle(color=rivercolors)

#     # Loop through each river to plot the elements in the same color.
#     for r, color in zip(rivers, rivercolors):
#         # -- Fill the confidence interval for this river --
#         ax.fill_between(date_idx,
#                         cumboot_qs.sel(chem=c, river=r, quantile="0.05"),
#                         cumboot_qs.sel(chem=c, river=r, quantile="0.95"),
#                         alpha=0.3,
#                         color=color)

#         # -- Plot the median/actual scaled cumulative flux as a line --
#         ax.plot(date_idx,
#                 cumboot_qs.sel(chem=c, quantile="actual", river=r),
#                 color=color,
#                 label=f"{r}")  # Label each river; adjust as needed

#         # -- Plot sample points for this river --
#         # Here we select sample points where data is present.
#         da_points = cumboot_qs.sel(chem=c, quantile="actual", river=r).where(
#                         all_samples.sel(chem=c, river=r).notnull())
#         da_points.plot.scatter(x="Date",
#                                ax=ax,
#                                add_colorbar=False,
#                                color=color)

#     # Remove any automatically generated title.
#     ax.set_title("")
#     # Remove x-axis label.
#     ax.set_xlabel("")
#     # Set the left y-axis label.
#     ax.set_ylabel(f"Cumulative {c} Flux (tonnes)", fontsize=24)

#     # Format the dates on the x-axis.
#     pretty_dates(ax, date_index=water_year)

# # Remove legend from the plotted axes (we will add a consolidated legend later).
# for ax in axes.flat:
#     leg = ax.get_legend()
#     if leg is not None:
#         leg.remove()

# # Hide x-axis tick labels for all subplots except the bottom row.
# for ax in axes[:-1, :].flat:
#     ax.tick_params(labelbottom=False)

# # Now, ensure the subplot in the second row, second column has its tick labels visible.
# axes[1, 1].tick_params(labelbottom=True)
# pretty_dates(axes[1, 1], date_index=water_year)

# ## Insert for secondary y-axis labels.
# tol = 1e-4  # Tolerance for comparing positions
# # Loop over each primary axis that contains data.
# for primary_ax in axes.flat[:-1]:
#     # Look for its twin axis (created in plot_river_background).
#     for ax_candidate in fig.axes:
#         if ax_candidate is primary_ax:
#             continue  # Skip the primary axis itself.
#         pos_primary = primary_ax.get_position()
#         pos_candidate = ax_candidate.get_position()
#         if (abs(pos_primary.x0 - pos_candidate.x0) < tol and
#             abs(pos_primary.y0 - pos_candidate.y0) < tol and
#             abs(pos_primary.width - pos_candidate.width) < tol and
#             abs(pos_primary.height - pos_candidate.height) < tol):
#             # Set the secondary y-axis label.
#             ax_candidate.set_ylabel("Streamflow (m³/s)", fontsize=26, labelpad=10)
#             break  # Move to the next primary axis.
# ###

# # Re-enable x-axis tick marks (only at the bottom) on all subplots.
# for ax in axes.flat:
#     ax.tick_params(axis='x', which='both', bottom=True, top=False)

# # Use the empty (bottom right) subplot for a consolidated legend.
# legend_ax = axes.flat[-1]
# # Get the legend handles and labels from the first (non-empty) subplot.
# handles, labels = axes.flat[0].get_legend_handles_labels()
# legend_ax.legend(handles, labels, loc='center', frameon=True, edgecolor='gray')
# legend_ax.axis('off')  # Turn off the axis so only the legend shows.

# plt.savefig(f"{fig_out_root}unscaled_cumulative_flux_annual.jpeg")
# #####end chat format testing for scaled plot


In [ ]:
fluxbias_stats = all_errorstats.sel(dim_1="fluxBias.bias3").to_pandas()
fluxbias_stats.to_csv(csv_out_root+"fluxbias_stats.csv")
fluxbias_stats

In [ ]:
#list(filter(lambda ax: ax.get_label() == "<colorbar>", fig.axes))[0].remove()

In [ ]:


# import matplotlib.pyplot as plt
# import xarray as xr
# from load_rivers import *  # This imports variables like cumboot_qs, areas, rivers, chemicals, all_samples, rivercolors, river_cmap, pretty_dates, water_year, date_idx, etc.

# # Compute scaled cumulative flux.
# cumboot_scaled = cumboot_qs * (xr.DataArray(areas, coords={"river": rivers}))
# cumboot_scaled.attrs["units"] = "mt/km^2"
# cumboot_qs.attrs["units"] = "mt"

# # Create a single row of subplots, one per chemical.
# fig, axes = plt.subplots(nrows=1, ncols=len(chemicals),
#                          figsize=(24, 8),
#                          gridspec_kw={'wspace': 0.09},
#                          constrained_layout=True)
# fig.suptitle("", fontsize=16, y=0.91)

# for c, ax in zip(chemicals, axes):
#     # Set up the background.
#     plot_river_background(ax)

#     # Set the property cycle for consistent river colors.
#     ax.set_prop_cycle(color=rivercolors)

#     # Plot confidence intervals for each river.
#     for r, color in zip(rivers, rivercolors):
#         ax.fill_between(date_idx,
#                         cumboot_scaled.sel(chem=c, river=r, quantile="0.05"),
#                         cumboot_scaled.sel(chem=c, river=r, quantile="0.95"),
#                         alpha=0.3)

#     # Plot the actual scaled cumulative flux.
#     ax.plot(date_idx, cumboot_scaled.sel(chem=c, quantile="actual").transpose(), label=rivers)

#     # Plot sample points.
#     cumboot_scaled.sel(chem=c, quantile="actual").where(
#                   all_samples.sel(chem=c).notnull()
#               ).plot.scatter(x="Date",
#                              hue="river",
#                              cmap=river_cmap,
#                              ax=ax,
#                              add_colorbar=False)

#     ax.set_title(c)
#     ax.set_ylabel("Cumulative Flux (metric tonnes)")
#     ax.legend(loc="upper left")
#     pretty_dates(ax, date_index=water_year)

# plt.savefig('cumulative_flux_summary_scaled_only', bbox_inches='tight')
# plt.show()


In [ ]:
# # helper to boost saturation / darken
# def adjust_saturation(hex_color, sat_factor=1.4, light_factor=1.0):
#     """
#     Adjust saturation and lightness for a hex color.
#       - sat_factor > 1 boosts saturation
#       - light_factor < 1 darkens
#     """
#     r, g, b = to_rgb(hex_color)
#     h, l, s = colorsys.rgb_to_hls(r, g, b)
#     s = max(0, min(1, s * sat_factor))
#     l = max(0, min(1, l * light_factor))
#     r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)
#     return to_hex((r2, g2, b2))
